# Run all: Bronze -> Silver -> Gold in one click

Use this for every normal load.
1. Run the first code cell. Three boxes appear at the top.
2. Fill them in (normally `load_type = incremental` and the date of the folder you downloaded; leave the folder blank).
3. Run the other cells (or Run all).

Each stage starts only if the previous one succeeded. Setup is NOT needed here (one-time notebook).

In [ ]:
# CELL 1 - run this first. It only creates the three input boxes.
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

In [ ]:
from datetime import datetime, timezone

load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()
source_folder = dbutils.widgets.get("source_folder").strip()

# Decide the date once so every layer uses the same one
if load_type == "incremental" and not processing_date:
    processing_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    print("No processing_date given; using today (UTC):", processing_date)

params = {"load_type": load_type, "processing_date": processing_date, "source_folder": source_folder}
gold_params = {"load_type": load_type, "processing_date": processing_date}
print("Parameters:", params)

pipeline_started = datetime.now(timezone.utc)

print("\n=== BRONZE ===")
dbutils.notebook.run("./01_bronze_ingestion/00_run_bronze", 0, params)
print("Bronze done.")

print("\n=== SILVER ===")
dbutils.notebook.run("./02_silver_transformation/00_run_silver", 0, params)
print("Silver done.")

print("\n=== GOLD ===")
dbutils.notebook.run("./03_gold_aggregation/00_run_gold", 0, gold_params)
print("Gold done.")

print("\nPipeline finished in", round((datetime.now(timezone.utc) - pipeline_started).total_seconds() / 60, 1), "minutes")

In [ ]:
from pyspark.sql import functions as F

# Latest log rows for all three layers
display(
    spark.table("workspace.default.pipeline_execution_logs")
    .orderBy(F.col("end_timestamp").desc())
    .select("layer", "table_name", "load_type", "processing_date", "status",
            "rows_read", "rows_inserted", "rows_updated", "rows_quarantined",
            "start_timestamp", "end_timestamp")
    .limit(12)
)